## 0. Setup

In [20]:
import json
import sys
import warnings
from pathlib import Path
from typing import Any, Dict, List


import pandas as pd

warnings.filterwarnings('ignore')

root = Path('..').resolve()
sys.path.append(str(root / 'src'))

from config import DATA_DIR
from preprocessing.normalization import BibleReferenceNormalizer
from preprocessing.normalization.resolver import ExactBookMatcher, FuzzyBookMatcher
from bible_data import load_bible_data

print('Setup complete.')

Setup complete.


## 1. Load Data

In [2]:
bible_data = load_bible_data()
print(f'Loaded {len(bible_data['books'])} books.')

Loaded 66 books.


In [3]:
with open(DATA_DIR / 'gold_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_gold = pd.DataFrame(data)[['text', 'structures', 'norms']]
print(f'Test set: {len(df_gold)} messages, '
      f'{df_gold['structures'].apply(len).sum()} total gold references')
df_gold.head()

Test set: 60 messages, 93 total gold references


,text,structures,norms
0,Efesus 1-2 done,"[{'book_start': 'Efesus', 'start_chapter': 1, ...","[{'book_start': 'Efesus', 'start_chapter': 1, ..."
1,Why 1-2 done,"[{'book_start': 'Why', 'start_chapter': 1, 'bo...","[{'book_start': 'Wahyu', 'start_chapter': 1, '..."
2,Efs1-2 done,"[{'book_start': 'Efs', 'start_chapter': 1, 'bo...","[{'book_start': 'Efesus', 'start_chapter': 1, ..."
3,_Bil 1-2_ ✓,"[{'book_start': 'Bil', 'start_chapter': 1, 'bo...","[{'book_start': 'Bilangan', 'start_chapter': 1..."
4,Gal 3-6 done,"[{'book_start': 'Gal', 'start_chapter': 3, 'bo...","[{'book_start': 'Galatia', 'start_chapter': 3,..."


## 2. Normalization Evaluation

In [4]:
FIELDS = ['book_start', 'start_chapter', 'book_end', 'end_chapter']

def run_normalization(df: pd.DataFrame, use_fuzzy: bool) -> pd.DataFrame:
    norm = BibleReferenceNormalizer(use_fuzzy=use_fuzzy)
    rows = []
    for _, row in df.iterrows():
        predicted = norm.normalize(row['structures'])
        gold = row['norms']
        for i in range(max(len(gold), len(predicted))):
            g = gold[i] if i < len(gold) else None
            p = predicted[i] if i < len(predicted) else None
            rows.append({'text': row['text'], 'gold': g, 'predicted': p})
    return pd.DataFrame(rows), norm

print("Running exact+fuzzy...")
df_fuzzy, norm_fuzzy = run_normalization(df_gold, use_fuzzy=True)

print("Running exact-only...")
df_exact, norm_exact = run_normalization(df_gold, use_fuzzy=False)

print(f'\n{len(df_fuzzy)} reference pairs evaluated.')

Running exact+fuzzy...
Running exact-only...
2026-04-22 09:13:20 | WARNING  | bible_pipeline.preprocessing.normalization.resolver | No match found for: 'Zed'
2026-04-22 09:13:20 | WARNING  | bible_pipeline.preprocessing.normalization.normalizer | Skipping ref, unresolveable start book: 'Zed'
2026-04-22 09:13:20 | WARNING  | bible_pipeline.preprocessing.normalization.resolver | No match found for: '2 Raja-blraja'
2026-04-22 09:13:20 | WARNING  | bible_pipeline.preprocessing.normalization.normalizer | Skipping ref, unresolveable start book: '2 Raja-blraja'

93 reference pairs evaluated.


In [ ]:
def ref_match(pred, gold) -> bool:
    if pred is None or gold is None:
        return False
    return all(pred.get(f) == gold.get(f) for f in FIELDS)

def compute_metrics(df: pd.DataFrame) -> Dict[str, Any]:
    correct = df.apply(lambda r: ref_match(r['predicted'], r['gold']), axis=1)
    skipped = df[df['gold'].notna() & df['predicted'].isna()]

    field_scores = {}
    for f in FIELDS:
        sub = df[df['gold'].notna()]
        hits = sub.apply(lambda r: r['predicted'] is not None and r['predicted'].get(f) == r['gold'].get(f), axis=1)
        field_scores[f] = hits.sum() / len(sub)
    
    return {
        'accuracy': correct.sum() / len(correct),
        'correct': int(correct.sum()),
        'total': len(correct),
        'skipped': len(skipped),
        'field_scores': field_scores,
        'correct_mask': correct,
    }

m_fuzzy = compute_metrics(df_fuzzy)
m_exact = compute_metrics(df_exact)

# Summary Table
summary = pd.DataFrame({
    'Exact-only': [m_exact['accuracy'], m_exact['correct'], m_exact['skipped']],
    'Exact+Fuzzy': [m_fuzzy['accuracy'], m_fuzzy['correct'], m_fuzzy['skipped']],
}, index=['Accuracy@1', 'Correct', 'Skipped'])

print("=== Ablation: Matching Strategy ===\n")
print(summary.to_string())
print(f"\nFuzzy contribution: +{m_fuzzy['correct'] - m_exact['correct']} references recovered\n")

print("=== Field-level Accuracy (Exact+Fuzzxy) ===\n")
for f, score in m_fuzzy['field_scores'].items():
    print(f"  {f:<20} {score:.3f}")

=== Ablation: Matching Strategy ===

            Exact-only  Exact+Fuzzy
Accuracy@1    0.978495          1.0
Correct      91.000000         93.0
Skipped       2.000000          0.0

Fuzzy contribution: +2 references recovered

=== Field-level Accuracy (Exact+Fuzzy) ===

  book_start           1.000
  start_chapter        1.000
  book_end             1.000
  end_chapter          1.000


## 3. Fuzzy Stress Test

### 3.1 Short Alias collisions

In [13]:
collision_candidates = [
    ('amo', 'Amos'),
    ('mal', 'Maleakhi'),
    ('fil', 'Filipi'),
    ('yoh', 'Yohanes'),
    ('rat', 'Ratapan'),
    ('dan', 'Daniel'),
]

fuzzy_matcher = FuzzyBookMatcher(bible_data['books'])
print('=== Collision Stress Test ===\n')
for query, expected in collision_candidates:
    result = fuzzy_matcher.match(query)
    resolved = result['name'] if result else None
    status = '✓' if resolved == expected else '✗  COLLISION'
    print(f'  {query!r:<10} → {resolved:<20} expected: {expected:<20} {status}') 

=== Collision Stress Test ===

  'amo'      → Amos                 expected: Amos                 ✓
  'mal'      → Maleakhi             expected: Maleakhi             ✓
  'fil'      → Filipi               expected: Filipi               ✓
  'yoh'      → Yohanes              expected: Yohanes              ✓
  'rat'      → Ratapan              expected: Ratapan              ✓
  'dan'      → Daniel               expected: Daniel               ✓


### 3.2  Typo Cases

In [18]:
typo_cases = [
    # 1 char swap
    ('kejdian', 'Kejadian'),
    ('mazmru', 'Mazmur'),
    ('ysaya', 'Yesaya'),

    # 1 char append
    ('keluaraan', 'Keluaran'),
    ('gaaltia', 'Galatia'),
    ('fiilipi', 'Filipi'),
    ('matiuss', 'Matius'),
    ('yudass', 'Yudas'),
    ('romaa', 'Roma'),

    # Char drop
    ('zakaria', 'Zakharia'),
    ('habakk', 'Habakuk'),
    ('yehezkl', 'Yehezkiel'),

    # complete garbage
    ('zzz', None),
    ('Zorgblat', None),
    ('buku', None),
    ('selesai', None),
]

print('=== Typo Tolerance Boundary ===\n')
print(f"  {'Query':<20} {'Resolved':<22} {'Expected':<22} {'Desc':<40} Status")
print(f"  {'-'*18} {'-'*20} {'-'*20} {'-'*38} ------")

pass_count = fail_count = 0
for query, expected in typo_cases:
    result = fuzzy_matcher.match(query)
    resolved = result['name'] if result else None

    if expected is None:
        ok = resolved is None
        status = '✓ correctly rejected' if ok else '✗ FALSE POSITIVE'
    else:
        ok = resolved == expected
        status = '✓' if ok else '✗'

    if ok:
        pass_count += 1
    else:
        fail_count += 1
    
    print(f'  {query!r:<10} → {str(resolved):<22} expected: {str(expected):<22} {status}')

print(f'\n  Passed: {pass_count}/{len(typo_cases)}  |  Failed: {fail_count}/{len(typo_cases)}')

=== Typo Tolerance Boundary ===

  Query                Resolved               Expected               Desc                                     Status
  ------------------ -------------------- -------------------- -------------------------------------- ------
  'kejdian'  → Kejadian               expected: Kejadian               ✓
  'mazmru'   → Mazmur                 expected: Mazmur                 ✓
  'ysaya'    → Ayub                   expected: Yesaya                 ✗
  'keluaraan' → Keluaran               expected: Keluaran               ✓
  'gaaltia'  → Galatia                expected: Galatia                ✓
  'fiilipi'  → Filipi                 expected: Filipi                 ✓
  'matiuss'  → Matius                 expected: Matius                 ✓
  'yudass'   → Yudas                  expected: Yudas                  ✓
  'romaa'    → Roma                   expected: Roma                   ✓
  'zakaria'  → Zakharia               expected: Zakharia               ✓
  'habakk'

### 3.3 Prefix ambiguity

In [22]:
prefix_cases = [
    ('samuel', None),
    ('korintus', None),
    ('petrus', None),
    ('yohanes', 'Yohanes'),
]

print('=== Prefix Ambiguity ===\n')
exact_matcher = ExactBookMatcher(bible_data['books'])
for query, expected in prefix_cases:
    e = exact_matcher.match(query)
    f = fuzzy_matcher.match(query)
    print(f"  {query!r:<12}  exact→ {str(e['name']) if e else 'None':<20}  fuzzy→ {str(f['name']) if f else 'None'}")

=== Prefix Ambiguity ===

  'samuel'      exact→ None                  fuzzy→ 1 Samuel
  'korintus'    exact→ None                  fuzzy→ 1 Korintus
  'petrus'      exact→ None                  fuzzy→ 1 Petrus
  'yohanes'     exact→ Yohanes               fuzzy→ Yohanes
